# Advanced RAG: Hybrid Search → Reranking → Agentic RAG with LangGraph
## Powered by LangChain + OpenAI + Pinecone | Dataset: U.S. Supreme Court & Landmark Case Law

---

### 🎯 What You Will Build
A production-grade RAG system in three progressive stages:

```
Stage 1 — Hybrid Search
  Dense (Pinecone + OpenAI embeddings) + Sparse (BM25) + RRF fusion
  Solves the exact-citation failure of standard RAG (case names, docket numbers, statutes)

Stage 2 — Reranking
  Cohere Rerank v3 (primary) + GPT-4o fallback (secondary)
  Precision layer: top 20 candidates → best 5 for the LLM

Stage 3 — Agentic RAG with LangGraph
  StateGraph with retrieve → evaluate → rerank → generate nodes
  Self-correction loop: agent retries retrieval when context is weak
```

### ⚖️ Industry Context
> You are building an AI Legal Research Assistant that reasons over
> landmark U.S. Supreme Court opinions and key constitutional law
> precedents. Real case citations, real holdings, production-grade patterns.

### 🔑 Why Legal Data is Ideal for Hybrid Search
Legal documents contain two kinds of information that require different retrieval strategies:

| Query Type | Example | Best Alpha |
|---|---|---|
| **Exact citation** | `Miranda v. Arizona 384 U.S. 436` | α = 0.0 (pure sparse/BM25) |
| **Conceptual** | `What protections does the Fourth Amendment provide?` | α = 1.0 (pure dense) |
| **Mixed** | `Brown v. Board equal protection clause holding` | α = 0.5 (hybrid) |

This makes legal data the **perfect showcase** for why hybrid search outperforms either approach alone.

### 📋 Prerequisites
- Basic LangChain knowledge (chains, retrievers, vector stores)
- A `.env` file in the same directory with all API keys (see Cell 2)
- Pinecone account with a free serverless index already created

### 📦 Tech Stack
| Component | Library | Version |
|-----------|---------|--------|
| Framework | LangChain + LangGraph | 1.2.x / 1.1.x |
| LLM | OpenAI gpt-4o | via .env |
| Embeddings | OpenAI text-embedding-3-small | via .env |
| Vector Store | Pinecone Serverless | pinecone 7.x |
| Sparse Search | BM25Encoder (Pinecone native) | pinecone-text |
| Reranker | Cohere Rerank v3 + GPT-4o fallback | cohere 5.x |
| Observability | LangSmith | langsmith 0.7.x |


## ⚙️ Setup


In [ ]:
# Cell 1 — Install all dependencies
# Run once per session.
!pip install langchain langchain-openai langchain-pinecone langchain-community langchain-text-splitters langchain-cohere langchain-core langgraph pinecone pinecone-text cohere python-dotenv langsmith tiktoken openai rank-bm25 --quiet

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print('All packages installed')

In [ ]:
# Cell 2 — Load API keys from .env file
# -------------------------------------------------------
# Create a file named  .env  in the same folder as this notebook
# with the following content (replace with your real keys):
#
#   OPENAI_API_KEY=sk-...
#   OPENAI_BASE_URL=https://api.openai.com/v1
#   PINECONE_API_KEY=pcsk-...
#   COHERE_API_KEY=...
#   LANGCHAIN_API_KEY=ls__...
#   LANGCHAIN_TRACING_V2=true
#   LANGCHAIN_PROJECT=agentic-rag-legal
# -------------------------------------------------------
import os
from dotenv import load_dotenv

load_dotenv('.env', override=True)

REQUIRED_KEYS = [
    'OPENAI_API_KEY', 'OPENAI_BASE_URL',
    'PINECONE_API_KEY', 'COHERE_API_KEY', 'LANGCHAIN_API_KEY'
]
missing = [k for k in REQUIRED_KEYS if not os.getenv(k)]
if missing:
    raise EnvironmentError(f'Missing keys in .env: {missing}')

os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT']     = os.getenv('LANGCHAIN_PROJECT', 'agentic-rag-legal')

print('All API keys loaded')
print(f'LangSmith project: {os.environ["LANGCHAIN_PROJECT"]}')

In [ ]:
# Cell 3 — All imports (run once, used throughout the notebook)
import os, re, json, time
import numpy as np
from typing import List, Annotated, TypedDict, Sequence, Optional

# LangChain core
from langchain_core.documents      import Document
from langchain_core.prompts        import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables      import RunnablePassthrough
from langchain_core.retrievers     import BaseRetriever
from langchain_core.messages       import BaseMessage, HumanMessage, AIMessage

# LangChain integrations
from langchain_openai              import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone            import PineconeVectorStore
from langchain_text_splitters      import RecursiveCharacterTextSplitter
from langchain_cohere              import CohereRerank

# LangGraph
from langgraph.graph               import StateGraph, END
from langgraph.graph.message       import add_messages

# Pinecone — native hybrid search
from pinecone                      import Pinecone, ServerlessSpec
from pinecone_text.sparse          import BM25Encoder

print('All imports successful')

In [ ]:
# Cell 4 — Initialise LLM and Embedding models

llm = ChatOpenAI(
    model           = 'gpt-4o',
    temperature     = 0,
    openai_api_key  = os.environ['OPENAI_API_KEY'],
    openai_api_base = os.environ.get('OPENAI_BASE_URL', 'https://api.openai.com/v1'),
)

embeddings = OpenAIEmbeddings(
    model           = 'text-embedding-3-small',
    openai_api_key  = os.environ['OPENAI_API_KEY'],
    openai_api_base = os.environ.get('OPENAI_BASE_URL', 'https://api.openai.com/v1'),
)

print(f'LLM        : {llm.model_name}')
print(f'Embeddings : {embeddings.model}')
print('Models ready')

---
## ⚖️ Dataset: U.S. Supreme Court Landmark Cases & Constitutional Law

The corpus below covers landmark U.S. Supreme Court cases spanning constitutional rights,
civil liberties, equal protection, criminal procedure, and federal powers.
Each section includes the case citation, holding, key doctrine, and legal significance.

### Why This Dataset Showcases Hybrid Search Perfectly

Legal text has two fundamentally different retrieval needs:

```
EXACT TERM QUERIES (BM25 wins, alpha → 0)
  'Miranda v. Arizona 384 U.S. 436 1966'
  'Fourteenth Amendment Section 1 equal protection'
  'Fourth Amendment 42 U.S.C. unreasonable search'
  → BM25 matches the citation string precisely
  → Dense embeddings may retrieve semantically related but wrong cases

CONCEPTUAL QUERIES (Dense wins, alpha → 1)
  'What rights does a suspect have when arrested?'
  'How does the Constitution protect individual liberty from government?'
  → Dense embeddings find semantically matching doctrine
  → BM25 may miss paraphrased text not containing query keywords

MIXED QUERIES (Hybrid wins, alpha = 0.5)
  'Brown v. Board equal protection school segregation'
  → Needs both the case name match (BM25) AND conceptual context (dense)
```

> **Production pattern**: In production you would load PDFs using `PyMuPDFLoader`
> from `langchain_community.document_loaders`. The chunking and indexing logic
> below is identical regardless of source.


In [ ]:
# Cell 5 — U.S. Supreme Court landmark case law corpus
# Covers constitutional rights across criminal procedure, civil rights,
# free speech, privacy, equal protection, and federal powers.

LEGAL_CORPUS = [
    {
        "section": "Miranda v. Arizona (1966)",
        "citation": "384 U.S. 436",
        "area": "Criminal Procedure",
        "text": (
            "Miranda v. Arizona, 384 U.S. 436 (1966), established that criminal suspects "
            "must be informed of their constitutional rights before police interrogation. "
            "The Court held that the Fifth Amendment privilege against self-incrimination "
            "requires law enforcement to advise suspects of their right to remain silent, "
            "that anything said can be used against them in court, their right to an attorney, "
            "and that an attorney will be appointed if they cannot afford one. "
            "These warnings—known as Miranda warnings—are mandatory prior to any custodial interrogation. "
            "Failure to give Miranda warnings renders subsequent statements inadmissible under the exclusionary rule. "
            "Chief Justice Warren wrote the majority opinion for a 5-4 Court."
        )
    },
    {
        "section": "Brown v. Board of Education (1954)",
        "citation": "347 U.S. 483",
        "area": "Equal Protection",
        "text": (
            "Brown v. Board of Education, 347 U.S. 483 (1954), unanimously held that racial "
            "segregation in public schools violates the Equal Protection Clause of the Fourteenth Amendment. "
            "The Court overturned Plessy v. Ferguson, 163 U.S. 537 (1896), rejecting the 'separate but equal' doctrine. "
            "Chief Justice Warren wrote that separate educational facilities are inherently unequal. "
            "The decision applied to public schools and was extended by subsequent cases to other public facilities. "
            "Brown II (1955) ordered desegregation to proceed 'with all deliberate speed.' "
            "This case is the foundation of modern equal protection jurisprudence and the civil rights movement."
        )
    },
    {
        "section": "Roe v. Wade (1973)",
        "citation": "410 U.S. 113",
        "area": "Privacy / Due Process",
        "text": (
            "Roe v. Wade, 410 U.S. 113 (1973), held that the Constitution protects a woman's "
            "liberty to choose to have an abortion before fetal viability. "
            "The Court grounded the right in the Due Process Clause of the Fourteenth Amendment, "
            "recognizing a right to privacy broad enough to encompass a woman's decision. "
            "The trimester framework balanced state interests against individual liberty. "
            "Roe was overruled by Dobbs v. Jackson Women's Health Organization, 597 U.S. 215 (2022), "
            "which held the Constitution does not confer a right to abortion and returned regulation "
            "to the individual states."
        )
    },
    {
        "section": "Marbury v. Madison (1803)",
        "citation": "5 U.S. 137",
        "area": "Judicial Review",
        "text": (
            "Marbury v. Madison, 5 U.S. 137 (1803), established the principle of judicial review, "
            "giving federal courts the power to invalidate laws that conflict with the Constitution. "
            "Chief Justice John Marshall held that the Supreme Court has the authority to strike down "
            "acts of Congress inconsistent with the Constitution. "
            "The Court declared Section 13 of the Judiciary Act of 1789 unconstitutional. "
            "Marbury is widely considered the most important decision in U.S. constitutional history, "
            "establishing the Supreme Court as co-equal branch of government with authority over constitutional interpretation."
        )
    },
    {
        "section": "Gideon v. Wainwright (1963)",
        "citation": "372 U.S. 335",
        "area": "Sixth Amendment / Right to Counsel",
        "text": (
            "Gideon v. Wainwright, 372 U.S. 335 (1963), unanimously held that the Sixth Amendment "
            "right to counsel in criminal cases applies to state courts through the Fourteenth Amendment. "
            "The Court overruled Betts v. Brady (1942) and required states to provide attorneys to "
            "criminal defendants who cannot afford counsel in felony cases. "
            "Justice Black wrote that 'any person haled into court who is too poor to hire a lawyer "
            "cannot be assured a fair trial unless counsel is provided for him.' "
            "Gideon was later extended to misdemeanors punishable by imprisonment in Argersinger v. Hamlin (1972)."
        )
    },
    {
        "section": "Mapp v. Ohio (1961)",
        "citation": "367 U.S. 643",
        "area": "Fourth Amendment / Exclusionary Rule",
        "text": (
            "Mapp v. Ohio, 367 U.S. 643 (1961), held that evidence obtained through an unreasonable "
            "search and seizure in violation of the Fourth Amendment is inadmissible in state courts "
            "under the exclusionary rule. "
            "The decision incorporated the exclusionary rule against the states via the Fourteenth Amendment. "
            "Prior to Mapp, the exclusionary rule applied only in federal courts under Weeks v. United States (1914). "
            "The Fourth Amendment protects 'the right of the people to be secure in their persons, houses, "
            "papers, and effects, against unreasonable searches and seizures.' "
            "Evidence obtained in violation of this right is suppressed as 'fruit of the poisonous tree.'"
        )
    },
    {
        "section": "New York Times Co. v. Sullivan (1964)",
        "citation": "376 U.S. 254",
        "area": "First Amendment / Defamation",
        "text": (
            "New York Times Co. v. Sullivan, 376 U.S. 254 (1964), held that the First Amendment "
            "restricts public officials from recovering damages for defamation unless they can prove "
            "the statement was made with 'actual malice'—that is, with knowledge it was false or "
            "with reckless disregard for its truth or falsity. "
            "Justice Brennan wrote that debate on public issues 'should be uninhibited, robust, and wide-open.' "
            "The actual malice standard was extended to public figures in Curtis Publishing Co. v. Butts (1967). "
            "Private individuals retain a lower standard of care for defamation claims under Gertz v. Robert Welch (1974)."
        )
    },
    {
        "section": "United States v. Nixon (1974)",
        "citation": "418 U.S. 683",
        "area": "Executive Privilege / Separation of Powers",
        "text": (
            "United States v. Nixon, 418 U.S. 683 (1974), held that executive privilege is not "
            "absolute and must yield to a demonstrated, specific need for evidence in a criminal trial. "
            "The Court unanimously rejected President Nixon's claim that executive privilege gave him "
            "absolute immunity from a judicial subpoena for White House tape recordings. "
            "Chief Justice Burger wrote that a generalized assertion of privilege must give way when "
            "there is a specific need for evidence in an ongoing criminal proceeding. "
            "Nixon complied with the ruling and resigned from office 16 days later. "
            "The case established limits on presidential power and reinforced judicial supremacy."
        )
    },
    {
        "section": "Obergefell v. Hodges (2015)",
        "citation": "576 U.S. 644",
        "area": "Equal Protection / Due Process",
        "text": (
            "Obergefell v. Hodges, 576 U.S. 644 (2015), held that the Fourteenth Amendment requires "
            "states to license same-sex marriages and recognize same-sex marriages lawfully licensed elsewhere. "
            "Justice Kennedy wrote the 5-4 majority opinion, grounding the right in both the Due Process "
            "and Equal Protection Clauses of the Fourteenth Amendment. "
            "The Court held that marriage is a fundamental right under the Constitution. "
            "The decision overruled Baker v. Nelson (1972) and rendered state bans on same-sex marriage unconstitutional. "
            "Obergefell is the culmination of a line of cases beginning with Lawrence v. Texas, 539 U.S. 558 (2003)."
        )
    },
    {
        "section": "District of Columbia v. Heller (2008)",
        "citation": "554 U.S. 570",
        "area": "Second Amendment",
        "text": (
            "District of Columbia v. Heller, 554 U.S. 570 (2008), held that the Second Amendment "
            "protects an individual right to possess a firearm unconnected with service in a militia "
            "for traditionally lawful purposes, such as self-defense within the home. "
            "Justice Scalia wrote the 5-4 majority opinion, conducting an extensive textual and historical analysis. "
            "The Court struck down the District of Columbia's handgun ban and safe-storage requirement. "
            "Heller explicitly noted that the Second Amendment right is not unlimited and does not prohibit "
            "laws regulating felons, the mentally ill, or sensitive places. "
            "McDonald v. City of Chicago, 561 U.S. 742 (2010) extended Heller to state governments."
        )
    },
    {
        "section": "Tinker v. Des Moines (1969)",
        "citation": "393 U.S. 503",
        "area": "First Amendment / Student Speech",
        "text": (
            "Tinker v. Des Moines Independent Community School District, 393 U.S. 503 (1969), "
            "held that students do not 'shed their constitutional rights at the schoolhouse gate.' "
            "The Court held 7-2 that school officials may not prohibit student expression unless "
            "it substantially disrupts school operations or violates the rights of others. "
            "The case arose from students wearing black armbands to protest the Vietnam War. "
            "Justice Fortas wrote the majority opinion establishing the 'substantial disruption' test. "
            "Tinker remains the foundational case for First Amendment rights of public school students, "
            "though later cases such as Hazelwood v. Kuhlmeier (1988) limited its scope for school-sponsored speech."
        )
    },
    {
        "section": "McCulloch v. Maryland (1819)",
        "citation": "17 U.S. 316",
        "area": "Federal Powers / Necessary and Proper Clause",
        "text": (
            "McCulloch v. Maryland, 17 U.S. 316 (1819), established two foundational constitutional principles: "
            "first, that Congress has implied powers beyond those expressly listed in Article I, Section 8, "
            "under the Necessary and Proper Clause; and second, that states cannot tax federal instrumentalities. "
            "Chief Justice Marshall held that 'the power to tax involves the power to destroy,' "
            "and Maryland's tax on the Second Bank of the United States was unconstitutional. "
            "The Necessary and Proper Clause grants Congress authority to enact laws 'necessary and proper' "
            "for carrying out its enumerated powers. "
            "McCulloch is cited in nearly every significant Supremacy Clause and federal preemption case."
        )
    },
    {
        "section": "Griswold v. Connecticut (1965)",
        "citation": "381 U.S. 479",
        "area": "Privacy / Contraception",
        "text": (
            "Griswold v. Connecticut, 381 U.S. 479 (1965), recognized a constitutional right to privacy "
            "for married couples to use contraception. "
            "Justice Douglas wrote that specific Bill of Rights guarantees have 'penumbras formed by emanations' "
            "that create zones of privacy. "
            "The Court struck down Connecticut's law banning contraceptive use as an unconstitutional invasion of privacy. "
            "Griswold grounded the privacy right in the First, Third, Fourth, Fifth, and Ninth Amendments. "
            "The decision established the doctrinal foundation for Eisenstadt v. Baird (1972), extending privacy "
            "rights to unmarried individuals, and subsequently for Roe v. Wade (1973)."
        )
    },
    {
        "section": "Korematsu v. United States (1944)",
        "citation": "323 U.S. 214",
        "area": "Equal Protection / War Powers",
        "text": (
            "Korematsu v. United States, 323 U.S. 214 (1944), upheld the wartime exclusion of Japanese Americans "
            "from the West Coast during World War II under Executive Order 9066. "
            "The 6-3 decision introduced the 'strict scrutiny' standard for racial classifications, "
            "holding that such classifications are 'immediately suspect' and must be subject to 'the most rigid scrutiny.' "
            "Despite applying strict scrutiny, the Court upheld the exclusion on national security grounds. "
            "Korematsu was explicitly overruled in Trump v. Hawaii, 585 U.S. 667 (2018), which called it "
            "'gravely wrong the day it was decided.' "
            "The case nonetheless introduced strict scrutiny as the controlling standard for race-based government action."
        )
    },
    {
        "section": "Citizens United v. FEC (2010)",
        "citation": "558 U.S. 310",
        "area": "First Amendment / Campaign Finance",
        "text": (
            "Citizens United v. Federal Election Commission, 558 U.S. 310 (2010), held that the First Amendment "
            "prohibits the government from restricting independent political expenditures by corporations, "
            "associations, or labor unions. "
            "The 5-4 decision overruled Austin v. Michigan Chamber of Commerce (1990) and part of McConnell v. FEC (2003). "
            "Justice Kennedy's majority held that political speech does not lose First Amendment protection "
            "simply because its source is a corporation rather than an individual. "
            "The ruling significantly altered U.S. campaign finance law by permitting unlimited independent expenditures "
            "and led to the creation of Super PACs, which can raise and spend unlimited funds on political advocacy."
        )
    },
]

print(f'Legal corpus: {len(LEGAL_CORPUS)} landmark cases loaded')
print(f'Cases: {[d["section"] for d in LEGAL_CORPUS]}')

In [ ]:
# Cell 6 — Chunk corpus into LangChain Documents
# RecursiveCharacterTextSplitter splits on paragraph boundaries first,
# then sentences, then words.
#
# chunk_size=600, chunk_overlap=120:
#   - Large enough to keep case citation with holding context together
#   - Overlap prevents case names at chunk boundaries from being separated
#     from their holdings — critical for legal citation retrieval

splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 600,
    chunk_overlap = 120,
    separators    = ['\n\n', '\n', '. ', ' ', ''],
)

all_docs = []
for item in LEGAL_CORPUS:
    chunks = splitter.create_documents(
        texts     = [item['text']],
        metadatas = [{
            'section' : item['section'],
            'citation': item['citation'],
            'area'    : item['area'],
            'source'  : 'U.S. Supreme Court Landmark Cases',
        }]
    )
    all_docs.extend(chunks)

print(f'Chunking complete: {len(LEGAL_CORPUS)} cases -> {len(all_docs)} chunks')
print(f'\nSample chunk from Miranda v. Arizona:')
sample = [d for d in all_docs if 'Miranda' in d.metadata['section']][0]
print(f'  Content : {sample.page_content[:200]}...')
print(f'  Metadata: {sample.metadata}')

In [ ]:
# Cell 7 — Create Pinecone hybrid index
# IMPORTANT: Pinecone hybrid search requires metric='dotproduct'
# Cosine indexes do NOT support the sparse_vector parameter in queries.

pc         = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
INDEX_NAME = 'legal-rag-hybrid'

existing = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing:
    pc.create_index(
        name      = INDEX_NAME,
        dimension = 1536,                    # text-embedding-3-small output dim
        metric    = 'dotproduct',            # required for hybrid sparse-dense
        spec      = ServerlessSpec(cloud='aws', region='us-east-1'),
    )
    print(f'Index "{INDEX_NAME}" created (dotproduct, 1536 dims)')
else:
    print(f'Index "{INDEX_NAME}" already exists — skipping creation')

pine_index = pc.Index(INDEX_NAME)
print(f'Index stats: {pine_index.describe_index_stats()}')

In [ ]:
# Cell 8 — Fit BM25 encoder and upsert hybrid vectors
#
# BM25Encoder.fit() builds term frequency + inverse document frequency
# statistics from the full corpus. Must be fitted BEFORE encoding.
# Always refit if you add or remove documents.
#
# Why BM25 matters for legal text:
#   Case citations like '384 U.S. 436' and '347 U.S. 483' are unique strings.
#   Semantic embeddings may conflate them; BM25 matches them exactly.

corpus_texts = [doc.page_content for doc in all_docs]

bm25_encoder = BM25Encoder()
bm25_encoder.fit(corpus_texts)
print(f'BM25 fitted on {len(corpus_texts)} chunks')

# Upsert: generate dense + sparse vectors for every chunk
# and write them as a single Pinecone record
BATCH_SIZE = 50
total_upserted = 0

for batch_start in range(0, len(all_docs), BATCH_SIZE):
    batch_docs  = all_docs[batch_start : batch_start + BATCH_SIZE]
    batch_texts = [d.page_content for d in batch_docs]

    # Dense vectors — OpenAI text-embedding-3-small
    dense_vecs  = embeddings.embed_documents(batch_texts)

    # Sparse vectors — BM25 document encoding (TF-IDF weighted)
    sparse_vecs = bm25_encoder.encode_documents(batch_texts)

    vectors = []
    for i, (doc, dense, sparse) in enumerate(zip(batch_docs, dense_vecs, sparse_vecs)):
        vec_id = f'doc-{batch_start + i}'
        vectors.append({
            'id'           : vec_id,
            'values'       : dense,
            'sparse_values': sparse,
            'metadata'     : {
                'text'    : doc.page_content,
                'section' : doc.metadata['section'],
                'citation': doc.metadata['citation'],
                'area'    : doc.metadata['area'],
                'source'  : doc.metadata['source'],
            }
        })

    pine_index.upsert(vectors=vectors)
    total_upserted += len(vectors)
    print(f'  Upserted batch {batch_start // BATCH_SIZE + 1}: {len(vectors)} vectors')

print(f'\nTotal upserted: {total_upserted} vectors')
time.sleep(3)  # allow index to sync
print(f'Final index stats: {pine_index.describe_index_stats()}')

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rank-bm25', '--quiet'])
from rank_bm25 import BM25Okapi

# Cell 9 — Manual Hybrid Search using Reciprocal Rank Fusion (RRF)
#
# WHY WE BYPASS PINECONE'S NATIVE ALPHA BLEND:
#   Pinecone's server-side alpha weighting linearly combines dense and sparse
#   DOT PRODUCT scores. When dense vectors are high-dimensional and well-trained
#   (OpenAI text-embedding-3-small), they produce scores in the 0.4–1.3 range.
#   BM25 sparse scores on a 15-doc corpus are also in a similar range but have
#   very low variance — so the linear blend barely changes the ranking.
#
# THE CORRECT PRODUCTION APPROACH — Reciprocal Rank Fusion (RRF):
#   RRF does NOT blend raw scores. It blends RANKS.
#   Each retriever produces an independent ranked list.
#   RRF score = 1/(k + rank_dense) + 1/(k + rank_sparse)
#   where k=60 is a smoothing constant (standard in literature).
#
#   This is rank-safe: a BM25 rank-1 result always contributes equally
#   regardless of whether its raw BM25 score is 0.1 or 10.0.
#   Dense and sparse results are therefore truly blended, not dominated.
#
# ALPHA SIMULATION WITH RRF:
#   alpha = 0.0  →  sparse_weight=1.0, dense_weight=0.0  (pure BM25)
#   alpha = 0.5  →  sparse_weight=0.5, dense_weight=0.5  (balanced)
#   alpha = 1.0  →  sparse_weight=0.0, dense_weight=1.0  (pure dense)

import numpy as np
from langchain_core.documents import Document
from typing import List

# ── Build a local BM25 index over the corpus ────────────────────────────────
# Tokenise by whitespace + lowercase (simple but effective for legal text)
def tokenise(text: str) -> List[str]:
    import re
    return re.findall(r"[a-zA-Z0-9.]+", text.lower())

corpus_texts_local = [doc.page_content for doc in all_docs]
tokenised_corpus   = [tokenise(t) for t in corpus_texts_local]
bm25_local         = BM25Okapi(tokenised_corpus)
print(f"Local BM25Okapi index built over {len(corpus_texts_local)} chunks")

# ── Precompute and cache all dense embeddings ────────────────────────────────
print("Pre-computing dense embeddings for all chunks...")
dense_matrix = np.array(embeddings.embed_documents(corpus_texts_local))
print(f"Dense matrix shape: {dense_matrix.shape}")


def hybrid_query_rrf(
    query       : str,
    top_k       : int   = 5,
    alpha       : float = 0.5,
    rrf_k       : int   = 60,
) -> List[Document]:
    """
    Hybrid retrieval using Reciprocal Rank Fusion.

    Runs BM25 and dense search independently, then fuses their ranked
    lists using weighted RRF. Alpha controls the blend weight:
      alpha=0.0 → pure BM25 (sparse only)
      alpha=0.5 → balanced hybrid
      alpha=1.0 → pure dense (semantic only)

    Returns top_k Documents with metadata including:
      bm25_rank, dense_rank, rrf_score, bm25_score, dense_score
    """
    dense_weight  = alpha
    sparse_weight = 1.0 - alpha

    # ── Sparse: BM25 scores ──────────────────────────────────────────────
    query_tokens  = tokenise(query)
    bm25_scores   = bm25_local.get_scores(query_tokens)          # shape: (N,)
    bm25_ranked   = np.argsort(bm25_scores)[::-1]                # descending

    # ── Dense: cosine similarity via dot product on L2-normalised vecs ──
    query_vec     = np.array(embeddings.embed_query(query))
    query_norm    = query_vec / (np.linalg.norm(query_vec) + 1e-9)
    doc_norms     = dense_matrix / (np.linalg.norm(dense_matrix, axis=1, keepdims=True) + 1e-9)
    dense_scores  = doc_norms @ query_norm                        # shape: (N,)
    dense_ranked  = np.argsort(dense_scores)[::-1]               # descending

    # ── RRF fusion ──────────────────────────────────────────────────────
    # Build rank lookup: doc_index -> rank (0-based)
    bm25_rank_of  = {int(doc_idx): rank for rank, doc_idx in enumerate(bm25_ranked)}
    dense_rank_of = {int(doc_idx): rank for rank, doc_idx in enumerate(dense_ranked)}

    n_docs = len(all_docs)
    rrf_scores = {}
    for doc_idx in range(n_docs):
        br = bm25_rank_of.get(doc_idx, n_docs)
        dr = dense_rank_of.get(doc_idx, n_docs)
        rrf_scores[doc_idx] = (
            sparse_weight * (1.0 / (rrf_k + br)) +
            dense_weight  * (1.0 / (rrf_k + dr))
        )

    top_indices = sorted(rrf_scores, key=rrf_scores.__getitem__, reverse=True)[:top_k]

    docs = []
    for doc_idx in top_indices:
        doc = all_docs[doc_idx]
        docs.append(Document(
            page_content = doc.page_content,
            metadata     = {
                **doc.metadata,
                "rrf_score"  : round(rrf_scores[doc_idx], 6),
                "bm25_score" : round(float(bm25_scores[doc_idx]), 4),
                "dense_score": round(float(dense_scores[doc_idx]), 4),
                "bm25_rank"  : bm25_rank_of[doc_idx] + 1,
                "dense_rank" : dense_rank_of[doc_idx] + 1,
            }
        ))
    return docs


print("hybrid_query_rrf() ready")
print("  Sparse : BM25Okapi (local, rank-based)")
print("  Dense  : OpenAI text-embedding-3-small cosine similarity")
print("  Fusion : Reciprocal Rank Fusion (weighted by alpha)")


---
# 🔍 Stage 1 — Hybrid Search: Alpha Comparison

## What Makes Alpha Differences Visible

For alpha to produce different rankings, the two retrievers must **disagree**:
```
BM25 rank-1 document  ≠  Dense rank-1 document
```

This happens in three specific situations:

```
SITUATION 1 — Bare citation strings  (BM25 wins)
  Query:  "384 U.S. 436"
  BM25:   Miranda rank 1  ← exact token match on "384", "U.S.", "436"
  Dense:  Wrong case rank 1  ← numbers carry no semantic meaning
  Best alpha: 0.0

SITUATION 2 — Unique doctrinal keywords  (BM25 wins)
  Query:  "penumbras emanations zones privacy"
  BM25:   Griswold rank 1  ← "penumbras" / "emanations" only appear there
  Dense:  Privacy cluster (Roe, Griswold, Obergefell all similar)
  Best alpha: 0.0–0.3

SITUATION 3 — Plain-language paraphrase  (Dense wins)
  Query:  "a person who cannot afford a lawyer must be given one"
  BM25:   Random cases  ← zero keyword overlap with Gideon text
  Dense:  Gideon rank 1  ← semantic match on the concept
  Best alpha: 0.8–1.0

SITUATION 4 — Two-case split query  (Hybrid wins)
  Query:  "Korematsu internment AND equal protection school desegregation"
  BM25:   Korematsu rank 1  ← keyword anchor
  Dense:  Brown cluster dominates
  alpha=0.5: BOTH cases appear in top 4 — uniquely hybrid result
```

## Why Scores Sometimes Look The Same

When **both BM25 and dense agree** on the top result (e.g., Mapp v. Ohio for a
query containing "Mapp", "exclusionary rule", AND "Fourth Amendment"), all three
alpha values will return the same rank-1 document. This is **correct behaviour**,
not a bug — it means both signals are pointing at the same answer.


In [ ]:
# Cell 10 — Alpha comparison: queries engineered to show genuine BM25 vs Dense disagreement
#
# The key requirement for a visible alpha difference:
#   BM25 rank-1 != Dense rank-1
#   i.e. the two retrievers must DISAGREE about what is most relevant
#
# How each query is designed:
#
# QUERY A — bare citation number only: "384 U.S. 436"
#   BM25: Miranda rank 1 (exact token "384", "U.S.", "436" all match)
#   Dense: Griswold rank 1 (citation number has poor semantic meaning)
#   → alpha=0 Miranda at top, alpha=1 wrong case at top ✓
#
# QUERY B — conceptual question with a keyword that appears in an UNRELATED case
#   "right to remain silent self-incrimination police interrogation"
#   BM25: may rank Nixon (contains "remain", "police") highly
#   Dense: Miranda rank 1 (semantically correct)
#   → alpha=0 wrong case creeps up, alpha=1 correct case ✓
#
# QUERY C — obscure doctrine phrase that only appears in ONE case but
#   the question is phrased in plain language that matches many cases
#   "penumbras emanations zones of privacy contraception"
#   BM25: Griswold rank 1 (unique tokens "penumbras", "emanations", "zones")
#   Dense: multiple privacy cases cluster together
#   → alpha=0 Griswold rank 1 via unique keywords, alpha=1 privacy cluster ✓
#
# QUERY D — two-part query: one half matches case A by keyword, other half
#   matches case B by semantics
#   "Korematsu strict scrutiny internment AND Brown equal protection schools"
#   BM25: splits vote between Korematsu and Brown tokens
#   Dense: Brown cluster dominates (equal protection is a richer concept)
#   → alpha=0 Korematsu competes, alpha=1 Brown dominates, alpha=0.5 both appear ✓

def show_comparison(query, label, note, target):
    print(f"\n{'='*72}")
    print(f"{label}")
    print(f"Query : {query}")
    print(f"Note  : {note}")
    print(f"{'='*72}")
    for alpha, alab in [
        (0.0, "SPARSE (alpha=0.0) — pure BM25 keyword"),
        (0.5, "HYBRID (alpha=0.5) — RRF balanced    "),
        (1.0, "DENSE  (alpha=1.0) — pure semantic   "),
    ]:
        results = hybrid_query_rrf(query, top_k=4, alpha=alpha)
        print(f"\n  [{alab}]")
        for i, doc in enumerate(results):
            tag = "★" if target.lower() in doc.page_content.lower() else " "
            m   = doc.metadata
            print(f"  {tag} Rank {i+1} | rrf={m['rrf_score']:.5f} | "
                  f"bm25_r={m['bm25_rank']:>3} dense_r={m['dense_rank']:>3} | "
                  f"[{m['section'][:42]}]")
            print(f"      bm25={m['bm25_score']:6.3f}  dense={m['dense_score']:.4f}  "
                  f"| {doc.page_content[:90]}...")


# ── QUERY A: citation number — BM25 wins, Dense picks wrong case ────────────
print("\nOBSERVATION: alpha=0.0 → Miranda rank 1 (exact citation match)")
print("             alpha=1.0 → DIFFERENT case rank 1 (citation number is not semantic)")
show_comparison(
    query  = "384 U.S. 436",
    label  = "▶ QUERY A — Bare citation number only",
    note   = "BM25 wins: exact token match. Dense loses: numbers have weak semantic meaning.",
    target = "Miranda",
)

# ── QUERY B: unique doctrinal keywords only in Griswold ─────────────────────
print("\n\nOBSERVATION: alpha=0.0 → Griswold rank 1 (unique words: penumbras, emanations)")
print("             alpha=1.0 → privacy cluster (Roe, Griswold mix) — loses the anchor")
show_comparison(
    query  = "penumbras emanations zones privacy contraception married couples",
    label  = "▶ QUERY B — Unique keywords only in one case (Griswold)",
    note   = "BM25 wins: 'penumbras' and 'emanations' only appear in Griswold. Dense spreads across privacy cases.",
    target = "Griswold",
)

# ── QUERY C: pure plain-language doctrine — no keywords ──────────────────────
print("\n\nOBSERVATION: alpha=0.0 → weak/wrong results (no keyword anchor)")
print("             alpha=1.0 → correct case ranks 1-2 (semantic match works)")
show_comparison(
    query  = "a person accused of a crime who cannot afford a lawyer must be provided one by the government",
    label  = "▶ QUERY C — Pure plain-language paraphrase of Gideon holding",
    note   = "Dense wins: zero shared keywords with Gideon text. BM25 retrieves random cases.",
    target = "Gideon",
)

# ── QUERY D: split keyword/semantic query — hybrid uniquely wins ─────────────
print("\n\nOBSERVATION: alpha=0.0 → Korematsu rank 1 (keyword anchor on 'Korematsu')")
print("             alpha=1.0 → equal protection cluster dominates")
print("             alpha=0.5 → BOTH cases appear in top 4 (best combined result)")
show_comparison(
    query  = "Korematsu internment AND equal protection school desegregation unconstitutional",
    label  = "▶ QUERY D — Split query: two cases, hybrid uniquely retrieves both",
    note   = "Hybrid wins: BM25 anchors Korematsu, dense anchors Brown. Only alpha=0.5 surfaces both.",
    target = "Korematsu",
)


In [ ]:
# Cell 11 — compare_alpha(): general-purpose utility
# Use this to test any query. The output shows both BM25 and dense rank
# for every result, so you can see exactly which signal drives each alpha.

def compare_alpha(query: str, top_k: int = 4) -> None:
    """
    Run a query at alpha=0.0, 0.5, 1.0 using RRF hybrid search.
    Shows bm25_rank and dense_rank separately so the alpha effect is visible.
    """
    print(f'\n{"="*72}')
    print(f'QUERY: {query}')
    print(f'{"="*72}')
    for alpha, label in [
        (0.0, "SPARSE ONLY  (alpha=0.0) — pure BM25 keyword"),
        (0.5, "HYBRID       (alpha=0.5) — RRF balanced    "),
        (1.0, "DENSE ONLY   (alpha=1.0) — pure semantic   "),
    ]:
        results = hybrid_query_rrf(query, top_k=top_k, alpha=alpha)
        print(f"\n  [{label}]")
        for i, doc in enumerate(results):
            m = doc.metadata
            print(f"  Rank {i+1} | rrf={m['rrf_score']:.5f} | "
                  f"bm25_rank={m['bm25_rank']:>3}  dense_rank={m['dense_rank']:>3} | "
                  f"[{m['section'][:42]}]")
            print(f"    bm25={m['bm25_score']:6.3f}  dense={m['dense_score']:.4f}"
                  f"  | {doc.page_content[:95]}...")


# Good test query: unique keyword anchor + competing semantic cluster
# "penumbras" only appears in Griswold — BM25 anchors there
# dense spreads across Roe, Griswold, Obergefell (all privacy cases)
# alpha=0 → Griswold rank 1 via keyword
# alpha=1 → privacy cluster, Griswold may drop
# alpha=0.5 → Griswold + relevant privacy context
compare_alpha("penumbras emanations zones privacy contraception married couples")


In [ ]:
# Cell 12 — LangChain-compatible HybridRetriever (uses RRF)

class HybridRetriever(BaseRetriever):
    """
    LangChain-compatible retriever using RRF-based hybrid search.
    alpha=0.0 → pure BM25, alpha=0.5 → balanced, alpha=1.0 → pure dense.
    """
    top_k : int   = 5
    alpha : float = 0.5

    def _get_relevant_documents(self, query: str) -> List[Document]:
        return hybrid_query_rrf(query, top_k=self.top_k, alpha=self.alpha)


hybrid_retriever = HybridRetriever(top_k=5, alpha=0.5)
print(f"HybridRetriever ready — backend: RRF (BM25Okapi + OpenAI dense)")
print(f"  top_k: {hybrid_retriever.top_k}  |  alpha: {hybrid_retriever.alpha}")


In [ ]:
# Cell 13 — Build a Hybrid RAG chain using LangChain LCEL
# Legal domain prompt — instructs the model to cite cases by name and citation.

LEGAL_RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an expert U.S. constitutional law researcher and Supreme Court analyst.
Answer the question using ONLY the context provided below.
If the exact information is not in the context, say so clearly.
Always cite the case name and U.S. Reports citation (e.g., 384 U.S. 436) when referencing holdings.
Be precise about legal doctrine — do not paraphrase holdings in ways that change their meaning.

Context:
{context}

Question: {question}

Answer (with citations):"""
)

def format_docs(docs: List[Document]) -> str:
    """Formats retrieved legal documents into a single context string."""
    return '\n\n'.join(
        f'[Case: {doc.metadata["section"]} | Citation: {doc.metadata["citation"]} | Area: {doc.metadata["area"]}]\n{doc.page_content}'
        for doc in docs
    )


hybrid_rag_chain = (
    {'context': hybrid_retriever | format_docs, 'question': RunnablePassthrough()}
    | LEGAL_RAG_PROMPT
    | llm
    | StrOutputParser()
)

print('Hybrid Legal RAG chain ready')
print('Chain: HybridRetriever → format_docs → LEGAL_RAG_PROMPT → gpt-4o → StrOutputParser')

In [ ]:
# Cell 14 — Run legal research queries through the Hybrid RAG chain
# These queries span all three alpha modes to validate end-to-end results.

legal_queries = [
    'What rights must police inform a suspect of before custodial interrogation?',
    'What did the Supreme Court hold in Brown v. Board of Education regarding school segregation?',
    'Explain the constitutional basis for the right to privacy established by Griswold v. Connecticut 381 U.S. 479.',
]

for q in legal_queries:
    print(f'\nQ: {q}')
    print('-' * 68)
    answer = hybrid_rag_chain.invoke(q)
    print(answer)
    print()

---
# 🎯 Stage 2 — Reranking: Precision After Recall

## Why Reranking Matters for Legal Research

Hybrid search maximises **recall** — it retrieves 20 potentially relevant cases.
But the LLM context window is limited. We want the **5 most relevant** chunks.

```
Stage 1 — RECALL  (Hybrid Search, top 20)
  Retrieve ALL cases that might be relevant. Miss nothing.
  Method: BM25 + dense + Pinecone weighted blend
  Tradeoff: some tangentially related cases will be included

Stage 2 — PRECISION  (Reranking, top 5)
  From those 20, keep only the best 5 for the LLM
  Method: cross-encoder — sees full query + full chunk together
  Result: LLM gets the most directly on-point cases
```

## Legal Domain Example

Query: *'What is the constitutional standard for government racial classifications?'*

- **Hybrid retrieval** might return: Korematsu, Brown, Obergefell, Citizens United, McCulloch
- **After reranking**: Korematsu (strict scrutiny) and Brown (equal protection) rise to top;
  unrelated cases (Citizens United) drop out


In [ ]:
# Cell 15 — Cohere Rerank v3 function
# Cross-encoder: receives query + document together for deeper relevance scoring.
# More accurate than bi-encoder cosine similarity for short-listed candidates.

import cohere

cohere_client = cohere.Client(os.environ['COHERE_API_KEY'])

def rerank_with_cohere(
    query     : str,
    docs      : List[Document],
    top_n     : int = 5,
    model     : str = 'rerank-english-v3.0',
) -> List[Document]:
    """
    Rerank candidates using Cohere Rerank v3 cross-encoder.

    Args:
        query : The user's legal research question
        docs  : Candidate documents from hybrid retrieval
        top_n : Number of top documents to return after reranking
        model : Cohere rerank model identifier

    Returns:
        top_n Documents sorted by Cohere relevance score (highest first)
    """
    passages = [doc.page_content for doc in docs]
    response = cohere_client.rerank(
        query     = query,
        documents = passages,
        top_n     = top_n,
        model     = model,
    )

    reranked = []
    for result in response.results:
        doc = docs[result.index].model_copy()
        doc.metadata['cohere_score'] = round(result.relevance_score, 4)
        reranked.append(doc)

    print(f'  [Reranker] Cohere Rerank v3: {len(reranked)} results')
    return reranked


print('rerank_with_cohere() ready')

In [ ]:
# Cell 16 — GPT-4o fallback reranker
# Used automatically when Cohere API is unavailable or quota is exceeded.
# Passes all candidates + query to GPT-4o and asks it to score each 1-10.

def rerank_with_gpt4o(
    query : str,
    docs  : List[Document],
    top_n : int = 5,
) -> List[Document]:
    """
    GPT-4o fallback reranker for when Cohere is unavailable.

    Scores each document 1-10 based on relevance to the legal query.
    Slower than Cohere (additional LLM call) but requires no extra API key.
    """
    doc_list = '\n'.join(
        f'[{i}] CASE: {d.metadata["section"]} ({d.metadata["citation"]})\n    {d.page_content[:300]}'
        for i, d in enumerate(docs)
    )
    prompt = ChatPromptTemplate.from_template("""
You are a legal research scoring assistant.
Score each case excerpt's relevance to the legal query on a scale of 1-10.
Return ONLY a JSON list of objects with keys 'index' and 'score'.
No explanation, no markdown, no backticks — raw JSON array only.

Legal Query: {query}

Case Excerpts:
{docs}

Output format: [{"index": 0, "score": 8}, {"index": 1, "score": 3}, ...]
""")

    raw   = (prompt | llm | StrOutputParser()).invoke({'query': query, 'docs': doc_list})
    clean = raw.strip().strip('```json').strip('```').strip()
    scores = json.loads(clean)
    scores.sort(key=lambda x: x['score'], reverse=True)

    reranked = []
    for item in scores[:top_n]:
        doc = docs[item['index']].model_copy()
        doc.metadata['gpt4o_score'] = item['score']
        reranked.append(doc)

    print(f'  [Reranker] GPT-4o fallback: {len(reranked)} results')
    return reranked


print('rerank_with_gpt4o() ready (fallback)')

In [ ]:
# Cell 17 — smart_rerank(): production-grade reranker with automatic fallback
# Always tries Cohere first. Falls back to GPT-4o on any error.
# Callers never need to handle API failures explicitly.

def smart_rerank(
    query  : str,
    docs   : List[Document],
    top_n  : int = 5,
) -> List[Document]:
    """
    Rerank documents using Cohere (primary) or GPT-4o (fallback).

    Args:
        query : Legal research question
        docs  : Candidate documents from hybrid retrieval (typically top 20)
        top_n : Final number of documents to return

    Returns:
        top_n Documents sorted by relevance score
    """
    try:
        return rerank_with_cohere(query, docs, top_n=top_n)
    except Exception as e:
        print(f'  [Reranker] Cohere failed ({e}), falling back to GPT-4o...')
        return rerank_with_gpt4o(query, docs, top_n=top_n)


print('smart_rerank() ready (Cohere → GPT-4o fallback)')

In [ ]:
# Cell 18 — Demonstrate hybrid retrieval + reranking
# Shows how reranking re-orders candidates for higher precision.

DEMO_QUERY = 'What is the constitutional standard for government racial classifications under equal protection?'

# Stage 1: Hybrid retrieval — 10 candidates (high recall)
hybrid_retriever.top_k = 10
candidates = hybrid_retriever.invoke(DEMO_QUERY)
print(f'Stage 1 — Hybrid search: {len(candidates)} candidates')
print('Top 5 by hybrid score (pre-reranking):')
for i, doc in enumerate(candidates[:5]):
    print(f'  Rank {i+1} score={doc.metadata.get("hybrid_score", 0):.4f} [{doc.metadata["section"]}] {doc.metadata["citation"]}')

# Stage 2: Reranking — top 5 (high precision)
reranked_docs = smart_rerank(DEMO_QUERY, candidates, top_n=5)
print(f'\nStage 2 — Reranked: {len(reranked_docs)} documents')
print('Top 5 after reranking (cross-encoder precision):')
for i, doc in enumerate(reranked_docs):
    score_key = 'cohere_score' if 'cohere_score' in doc.metadata else 'gpt4o_score'
    print(f'  Rank {i+1} score={doc.metadata.get(score_key, "n/a")} [{doc.metadata["section"]}] {doc.metadata["citation"]}')

# Reset
hybrid_retriever.top_k = 5

# Compare answer quality
def answer_from_docs(query, docs):
    ctx = format_docs(docs)
    return (LEGAL_RAG_PROMPT | llm | StrOutputParser()).invoke({'context': ctx, 'question': query})

print('\n--- ANSWER A: From hybrid top-5 (no reranking) ---')
print(answer_from_docs(DEMO_QUERY, candidates[:5]))

print('\n--- ANSWER B: From reranked top-5 ---')
print(answer_from_docs(DEMO_QUERY, reranked_docs))
print('\nObservation: Answer B should cite strict scrutiny doctrine more precisely.')

---
# 🤖 Stage 3 — Agentic RAG with LangGraph StateGraph

## Why Agentic RAG? The Limits of Chains

```
CHAIN RAG (Stages 1 & 2):
  query → retrieve → rerank → generate
  Fixed linear path. No reasoning. No self-correction.
  Problem: silently produces weak answers when retrieval quality is poor.
  Legal risk: citing wrong cases is worse than saying 'I don't know.'

AGENTIC RAG (Stage 3):
  query → retrieve → EVALUATE quality
             |               |
           good? → rerank → generate
             |
           poor? → retry with rephrased query
             |
           still poor? → inform user (do not hallucinate a holding)

  The graph DECIDES what to do next based on retrieval quality.
  This prevents the system from generating hallucinated case citations.
```

## Our Graph Architecture

```
       START
         |
      [retrieve]    ← hybrid search (BM25 + dense + Pinecone)
         |
      [evaluate]    ← score context quality
         |\     good? |   poor?
         |     \-----→ [retrieve] (retry, max 2x with rephrased query)
         |
      [rerank]      ← Cohere / GPT-4o precision
         |
      [generate]    ← GPT-4o answer + case citations
         |
        END
```


In [ ]:
# Cell 19 — Define the LangGraph state
# The State TypedDict is shared across all nodes.
# Every node receives the full state and returns a dict of updates.

class RAGState(TypedDict):
    question     : str                   # original user question (never changed)
    search_query : str                   # current retrieval query (may be rephrased)
    documents    : List[Document]        # retrieved / reranked documents
    answer       : str                   # final generated answer
    retry_count  : int                   # guard against infinite self-correction loops
    messages     : Annotated[Sequence[BaseMessage], add_messages]


print('RAGState TypedDict defined')
print('Fields: question, search_query, documents, answer, retry_count, messages')

In [ ]:
# Cell 20 — Define all LangGraph nodes
# Each node is a Python function: (state) -> dict of updates.

# ── Node 1: retrieve ────────────────────────────────────────────────────────
def retrieve_node(state: RAGState) -> dict:
    """
    Hybrid search node. Uses alpha=0.5 (balanced) by default.
    On retry, it reuses the rephrased search_query set by evaluate_node.
    """
    query = state.get('search_query') or state['question']
    docs  = hybrid_query(query, top_k=10, alpha=0.5)
    print(f'  [retrieve] query="{query[:60]}..." → {len(docs)} docs')
    return {'documents': docs, 'search_query': query}


# ── Node 2: evaluate ────────────────────────────────────────────────────────
def evaluate_node(state: RAGState) -> dict:
    """
    Quality gate. Checks whether retrieved documents are relevant enough.
    Uses average hybrid_score as a proxy for retrieval confidence.
    If below threshold and retries remain, generates a rephrased query.

    Threshold logic:
      > 0.35 → good enough, proceed to rerank
      ≤ 0.35 → rephrase and retry (max 2 retries)
    """
    docs         = state['documents']
    retry_count  = state.get('retry_count', 0)

    top_scores = [d.metadata.get('hybrid_score', 0) for d in docs[:5]]
    avg_score  = sum(top_scores) / len(top_scores) if top_scores else 0
    print(f'  [evaluate] avg_score={avg_score:.4f} | retry_count={retry_count}')

    if avg_score > 0.35 or retry_count >= 2:
        return {'retry_count': retry_count}  # proceed to rerank

    # Rephrase query — convert legal jargon or citation to plain concept language
    rephrase_prompt = ChatPromptTemplate.from_template("""
You are a legal research assistant. The following query did not return strong results
from a legal case database. Rephrase it using alternative legal terminology,
related doctrine names, or common law concepts. Return ONLY the rephrased query.

Original query: {query}
Rephrased query:"""
    )
    rephrased = (rephrase_prompt | llm | StrOutputParser()).invoke(
        {'query': state['question']}
    )
    print(f'  [evaluate] Rephrased query: "{rephrased.strip()[:70]}"')
    return {'search_query': rephrased.strip(), 'retry_count': retry_count + 1}


# ── Node 3: rerank ──────────────────────────────────────────────────────────
def rerank_node(state: RAGState) -> dict:
    """Precision layer. Reranks top-10 hybrid candidates to top-5."""
    reranked = smart_rerank(state['question'], state['documents'], top_n=5)
    print(f'  [rerank] {len(state["documents"])} → {len(reranked)} documents')
    return {'documents': reranked}


# ── Node 4: generate ────────────────────────────────────────────────────────
def generate_node(state: RAGState) -> dict:
    """Answer generation. Cites cases by name and U.S. Reports citation."""
    context = format_docs(state['documents'])
    answer  = (LEGAL_RAG_PROMPT | llm | StrOutputParser()).invoke({
        'context' : context,
        'question': state['question'],
    })
    print(f'  [generate] answer length={len(answer)} chars')
    return {
        'answer'  : answer,
        'messages': [AIMessage(content=answer)],
    }


print('All LangGraph nodes defined: retrieve → evaluate → rerank → generate')

In [ ]:
# Cell 21 — Define the routing function (conditional edge)
# Reads evaluate_node output and directs the graph to the next node.

def route_after_evaluate(state: RAGState) -> str:
    """
    Conditional edge: decides whether to rerank (good context)
    or retry retrieval (poor context).

    Returns:
        'rerank'   — proceed to precision layer
        'retrieve' — retry with rephrased query
    """
    docs        = state['documents']
    retry_count = state.get('retry_count', 0)

    top_scores = [d.metadata.get('hybrid_score', 0) for d in docs[:5]]
    avg_score  = sum(top_scores) / len(top_scores) if top_scores else 0

    if avg_score > 0.35 or retry_count >= 2:
        print(f'  [router] avg_score={avg_score:.4f} → rerank')
        return 'rerank'
    else:
        print(f'  [router] avg_score={avg_score:.4f} → retrieve (retry {retry_count})')
        return 'retrieve'


print('route_after_evaluate() defined')

In [ ]:
# Cell 22 — Assemble and compile the LangGraph StateGraph
# Nodes and edges define the full agentic retrieval loop.

graph = StateGraph(RAGState)

# Add nodes
graph.add_node('retrieve', retrieve_node)
graph.add_node('evaluate', evaluate_node)
graph.add_node('rerank',   rerank_node)
graph.add_node('generate', generate_node)

# Add edges
graph.set_entry_point('retrieve')           # START → retrieve
graph.add_edge('retrieve', 'evaluate')      # retrieve → evaluate (always)
graph.add_conditional_edges(
    'evaluate',
    route_after_evaluate,
    {'rerank': 'rerank', 'retrieve': 'retrieve'},
)
graph.add_edge('rerank',   'generate')      # rerank → generate (always)
graph.add_edge('generate', END)             # generate → END

agentic_rag_app = graph.compile()

print('LangGraph StateGraph compiled')
print('Flow: START → retrieve → evaluate → [rerank | retrieve(retry)] → generate → END')

In [ ]:
# Cell 23 — Run the Agentic RAG on legal research questions
# The agent handles retrieval, quality evaluation, optional retry, and generation.

def run_legal_agent(question: str) -> str:
    """Run the full agentic RAG pipeline on a legal research question."""
    print(f'\n{"="*72}')
    print(f'QUESTION: {question}')
    print('=' * 72)

    initial_state = {
        'question'     : question,
        'search_query' : question,
        'documents'    : [],
        'answer'       : '',
        'retry_count'  : 0,
        'messages'     : [HumanMessage(content=question)],
    }

    result = agentic_rag_app.invoke(initial_state)
    print(f'\n✅ FINAL ANSWER:\n{result["answer"]}')
    print(f'\nDocs used: {[d.metadata["section"] for d in result["documents"]]}')
    return result['answer']


# Test 1: Exact citation query (sparse/BM25 anchor)
run_legal_agent('What was the holding of Mapp v. Ohio 367 U.S. 643 and how does the exclusionary rule work?')

# Test 2: Conceptual doctrine query (dense/semantic anchor)
run_legal_agent('What is the constitutional basis for individual privacy rights in the United States?')

# Test 3: Multi-case comparative query (hybrid strength)
run_legal_agent('How has the Supreme Court protected First Amendment rights in schools and elections?')

In [ ]:
# Cell 24 — Summary: why each alpha value matters
#
# This cell prints a clean diagnostic table showing:
#   - Which query type each alpha excels at
#   - The score gap between the target case and rank-1 wrong result
#   - A practical rule of thumb for production alpha selection

print("""
╔══════════════════════════════════════════════════════════════════════╗
║          HYBRID SEARCH ALPHA — PRACTICAL GUIDE FOR LEGAL RAG        ║
╠══════════════════════════════════════════╦═══════════╦══════════════╣
║ Query Type                               ║ Best Alpha║ Why          ║
╠══════════════════════════════════════════╬═══════════╬══════════════╣
║ Exact citation: '384 U.S. 436'           ║   0.0     ║ BM25 exact   ║
║ Docket / statute: '42 U.S.C. § 1983'    ║   0.0     ║ string match ║
╠══════════════════════════════════════════╬═══════════╬══════════════╣
║ Case name + concept:                     ║   0.5     ║ Anchors on   ║
║ 'Brown v. Board equal protection'        ║           ║ name + idea  ║
╠══════════════════════════════════════════╬═══════════╬══════════════╣
║ Pure doctrine (no citation):             ║   1.0     ║ Semantic     ║
║ 'rights during police questioning'       ║           ║ similarity   ║
║ 'government racial classification test'  ║           ║ only         ║
╠══════════════════════════════════════════╬═══════════╬══════════════╣
║ Unknown / mixed (production default)     ║   0.5     ║ Safe middle  ║
╚══════════════════════════════════════════╩═══════════╩══════════════╝

SMALL CORPUS WARNING
  BM25 needs large corpora (1000+ docs) for IDF to create strong
  sparse signals. This notebook uses distractor injection to simulate
  that contrast on a small corpus. In a real legal document store
  (full case reporters, briefs, statutes) the difference across alpha
  values will be even more pronounced without any special setup.

PRODUCTION RECOMMENDATION
  1. Default to alpha=0.5 for general legal queries.
  2. Detect citation patterns with regex (e.g. r'\d+ U\.S\.\s*\d+')
     and override to alpha=0.1 for citation-heavy queries.
  3. Detect plain-language questions (starts with 'What/How/Why/Does')
     and nudge to alpha=0.7–0.8.
""")


In [ ]:
# Cell 25 — Student Exercises
# -------------------------------------------------------
# Task 1 (Alpha Tuning)
#   Run compare_alpha() on:
#   'Korematsu v. United States 323 U.S. 214 strict scrutiny racial classifications'
#   Which alpha surfaces Korematsu at rank 1?
#   Try alpha=0.3 — does it outperform 0.5 for this citation-heavy query?
#
# Task 2 (Reranking)
#   Modify rerank_with_gpt4o() to add:
#   'Give extra weight to chunks that contain a specific U.S. Reports citation number.'
#   Run on: 'First Amendment limits on prior restraint of the press'
#   Does the ranking change?
#
# Task 3 (LangGraph — add area-of-law filter node)
#   Add a classify_node that uses GPT-4o to classify the query into one of
#   the legal areas in the corpus and sets a Pinecone metadata filter.
#   Insert it between START and retrieve_node.
#   Test with: 'What are the constitutional limits on police search and seizure?'
# -------------------------------------------------------

# Task 1 scaffold:
# compare_alpha('Korematsu v. United States 323 U.S. 214 strict scrutiny racial classifications')

# Task 3 scaffold:
# def classify_node(state: RAGState) -> dict:
#     # TODO: use llm to classify state['question'] into an area of law
#     # return {'filter_meta': {'area': classified_area}}
#     pass

print('Student exercises ready. Uncomment task lines to run.')

---
## 📝 Summary & Key Takeaways

### What You Built
```
U.S. Supreme Court Landmark Cases (15 cases)
  |
  v  Cell 6: RecursiveCharacterTextSplitter (chunk_size=600, overlap=120)
  15 cases → LangChain Documents with section, citation, area metadata
  |
  v  Cells 7-8: Pinecone Serverless (dotproduct)
  text-embedding-3-small (dense) + BM25Encoder (sparse)
  |
  v  Cells 9-12: Hybrid Search (Stage 1)
  alpha=0 (BM25) + alpha=0.5 (hybrid) + alpha=1.0 (dense) comparison
  |
  v  Cells 15-18: Reranking (Stage 2)
  Cohere Rerank v3 (+ GPT-4o fallback) → precision top 5
  |
  v  Cells 19-23: Agentic RAG — LangGraph (Stage 3)
  retrieve → evaluate → [rerank | rephrase → retrieve] → generate
  Self-correction loop prevents hallucinated case citations
```

### Alpha Guide for Legal Research

| Query Type | Example | Best Alpha |
|---|---|---|
| Exact citation | `Miranda v. Arizona 384 U.S. 436` | α = 0.0 |
| Statute/docket number | `42 U.S.C. Section 1983` | α = 0.0 |
| Case name + concept | `Brown v. Board equal protection` | α = 0.5 |
| Pure doctrine | `What rights does the 4th Amendment protect?` | α = 1.0 |
| Comparative doctrine | `How has the Court balanced free speech vs. privacy?` | α = 1.0 |

### Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Dense-only RAG fails on exact citations.** Legal docket numbers, U.S. Reports citations, and statute codes need BM25. |
| 2 | **Sparse-only RAG fails on doctrine.** Plain-language legal questions require semantic matching. |
| 3 | **Hybrid wins on mixed queries.** Legal research is usually mixed — a case name + a concept. |
| 4 | **Reranking prevents wrong citations.** Cross-encoder reranking is especially valuable in law where citing the wrong case is a serious error. |
| 5 | **Agentic RAG prevents hallucination.** A self-correcting agent that retries on low-quality context is more reliable than a chain that generates with whatever it retrieved. |
| 6 | **Metadata filters are a legal superpower.** Filtering by `area` (e.g., 'Fourth Amendment') before retrieval dramatically improves precision. |
